# 🏥 ViBioMIR - Corpus Merge, Deduplication & Hugging Face Sync

Notebook này hỗ trợ quy trình chuẩn hóa toàn bộ kho ngữ liệu ViBioMIR:
1. **Kiểm tra trùng lặp (Overlap Analysis)**: Đối chiếu dữ liệu mới (JSONL shards) với dữ liệu đã tồn tại trên Hugging Face Storage Bucket.
2. **Làm sạch & Deduplication**: Loại bỏ nhiễu web (boilerplate, ads, navigation), chuẩn hóa Unicode NFC, sửa lỗi font/mojibake, và loại bỏ trùng lặp tuyệt đối theo `doc_id`.
3. **Tạo Shard Parquet chuẩn hóa**: Chia đều thành các shard ~100,000 dòng nén ZSTD level 9 tối ưu tốc độ đọc và dung lượng lưu trữ.
4. **Đồng bộ lên Hugging Face Storage Bucket**: Sử dụng `hf sync` đẩy lên `hf://buckets/nieths/ViBioMIR/corpus`.

## 1. Cài đặt Thư viện và Môi trường

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q pyarrow "huggingface_hub[cli]" tqdm loguru

# Cài đặt hf CLI chính thức (nếu chưa có)
!which hf || uv tool install hf

## 2. Cấu hình Đường dẫn và Token Hugging Face
> **Lưu ý quan trọng**: Để đồng bộ (upload) dữ liệu lên Bucket, `HF_TOKEN` phải có quyền **WRITE** (không dùng token quyền Read-only). Bạn có thể tạo tại: [Hugging Face Access Tokens](https://huggingface.co/settings/tokens).

In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi

# Đọc HF_TOKEN từ .env nếu chưa set trong environment
token = os.environ.get("HF_TOKEN")
if not token and Path(".env").exists():
    for line in Path(".env").read_text(encoding="utf-8").splitlines():
        if line.strip().startswith("HF_TOKEN="):
            token = line.split("=", 1)[1].strip()
            os.environ["HF_TOKEN"] = token
            break

api = HfApi(token=token)
user_info = api.whoami()
token_role = user_info.get("auth", {}).get("accessToken", {}).get("role", "unknown")
print(f"Logged in as: {user_info.get('name')} | Token role: {token_role}")
if token_role == "read":
    print("⚠️ CẢNH BÁO: Token hiện tại là quyền READ! Cần thay token quyền WRITE trong file .env để thực hiện sync/upload.")
else:
    print("✓ Token có quyền hợp lệ để upload dữ liệu.")

## 3. Kiểm tra Trùng lặp giữa Shard Mới và Bucket Cũ

In [ ]:
import json
import pyarrow.parquet as pq

bucket_id = "nieths/ViBioMIR"
old_parquet = Path("data/processed/parquet_corpus/corpus_part_000.parquet")

# Tải file cũ từ bucket nếu local chưa có
if not old_parquet.exists():
    print("Tải corpus_part_000.parquet từ Hugging Face Bucket...")
    old_parquet.parent.mkdir(parents=True, exist_ok=True)
    api.download_bucket_files(bucket_id=bucket_id, files=[("corpus/corpus_part_000.parquet", old_parquet)])

bucket_table = pq.read_table(old_parquet)
bucket_ids = set(bucket_table["doc_id"].to_pylist())
print(f"Số lượng tài liệu cũ trên Bucket: {len(bucket_ids):,}")

# Kiểm tra shard 4 và shard 5
for shard_name in ["crawled_shard_4.jsonl", "crawled_shard_5.jsonl"]:
    shard_path = Path("data/raw") / shard_name
    if not shard_path.exists():
        continue
    s_ids = set()
    with open(shard_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                d = json.loads(line)
                s_ids.add(int(d.get("doc_id") or d.get("id")))
    overlap = s_ids.intersection(bucket_ids)
    print(f"{shard_name}: {len(s_ids):,} unique IDs | Trùng với bucket cũ: {len(overlap):,} ({len(overlap)/len(s_ids)*100:.1f}%)")

## 4. Chạy Hợp nhất, Deduplicate và Tạo Shards Parquet (100k dòng/file)

In [ ]:
# Gọi trực tiếp script chuẩn hóa của dự án
!python scripts/merge_and_sync_corpus.py \
    --existing-parquet data/processed/parquet_corpus/corpus_part_000.parquet \
    --jsonl-files data/raw/crawled_shard_4.jsonl data/raw/crawled_shard_5.jsonl \
    --output-dir data/processed/parquet_corpus \
    --chunk-size 100000 \
    --min-chars 50

## 5. Kiểm tra tính toàn vẹn của các file Parquet đã tạo

In [ ]:
import pyarrow.parquet as pq

corpus_dir = Path("data/processed/parquet_corpus")
files = sorted(corpus_dir.glob("*.parquet"))

total_docs = 0
all_ids = set()

print(f"Tìm thấy {len(files)} shards:")
for f in files:
    t = pq.read_table(f)
    rows = len(t)
    total_docs += rows
    ids = t["doc_id"].to_pylist()
    all_ids.update(ids)
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  • {f.name}: {rows:,} dòng | Dung lượng: {size_mb:.2f} MB")

print("-" * 50)
print(f"Tổng số tài liệu: {total_docs:,}")
print(f"Số lượng doc_id duy nhất: {len(all_ids):,}")
assert total_docs == len(all_ids), "Phát hiện doc_id trùng lặp giữa các shard!"
print("✓ Toàn bộ dữ liệu nhất quán và không có trùng lặp!")

## 6. Đồng bộ lên Hugging Face Storage Bucket

Đồng bộ thư mục `data/processed/parquet_corpus` lên `hf://buckets/nieths/ViBioMIR/corpus`.

In [ ]:
# Chạy đồng bộ thông qua hf CLI
!bash scripts/sync_to_hf_bucket.sh